# Evaluation Lab — build your project's evaluation notebook

**AHLI Health AI Summer Camp 2026 · Day 3 workshop**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahli-org/ahli-summer-camp-website/blob/main/public/notebooks/evaluation-lab-template.ipynb)

> **How to use this notebook.** It is a *template*, not a finished analysis. Work
> through the sections on **your own project**: state your project in Section 0,
> build a data substrate (the synthetic generator below, or a public/real
> dataset), and adapt each move to your outcome type and decision. The
> intellectual work is *deciding what to build and how to interpret it* — use an
> AI coding assistant freely to scaffold the synthetic stand-in. Section 7
> converts your findings into your Project Workbook section.


## Section 0 — Project setup

*State your project in one or two sentences: the prediction/decision, the population, and the decision your model informs. Then build a data substrate. The synthetic generator below gives you ground-truth control; swap in the MIMIC-IV demo or your own dataset if that fits better.*

In [ ]:
import numpy as np
import pandas as pd

def make_synthetic(
    n=4000,
    prevalence=0.15,        # event rate of the positive class
    signal=1.2,             # strength of the genuine signal
    subgroup_gap=0.6,       # how much weaker the signal is in subgroup B
    shortcut_strength=2.0,  # a feature spuriously predictive in TRAIN only
    label_noise=0.05,       # fraction of labels flipped (uniform)
    shift=True,             # apply a train -> test distribution shift
    seed=0,
):
    """A binary-outcome stand-in with two subgroups, a planted shortcut, label
    noise, and an optional train->test shift. You control the ground truth, so
    you can demonstrate a metric reporting the wrong thing. Adapt the outcome
    type (multiclass / time-to-event / regression) to match YOUR project."""
    rng = np.random.default_rng(seed)
    subgroup = rng.integers(0, 2, n)                       # 0 = A, 1 = B
    x1 = rng.normal(0, 1, n)
    x2 = rng.normal(0, 1, n)
    # Genuine signal, weaker in subgroup B (a true performance gap):
    strength = signal * np.where(subgroup == 1, 1 - subgroup_gap, 1.0)
    logit = strength * x1 + 0.5 * x2 + np.log(prevalence / (1 - prevalence))
    p = 1 / (1 + np.exp(-logit))
    y = rng.binomial(1, p)
    # Label noise:
    flip = rng.random(n) < label_noise
    y = np.where(flip, 1 - y, y)
    # A shortcut: a feature correlated with y in TRAIN, broken at test time.
    split = rng.random(n) < 0.7
    shortcut = np.where(split, shortcut_strength * (y - 0.5), 0) + rng.normal(0, 1, n)
    df = pd.DataFrame({"x1": x1, "x2": x2, "shortcut": shortcut,
                       "subgroup": subgroup, "y": y, "is_train": split})
    train, test = df[df.is_train].copy(), df[~df.is_train].copy()
    if shift:  # shift the test distribution so a fragile model degrades
        test["x1"] = test["x1"] + 0.8
    return train.drop(columns="is_train"), test.drop(columns="is_train")

train_df, test_df = make_synthetic()
FEATURES = ["x1", "x2", "shortcut"]
X_train, y_train = train_df[FEATURES], train_df["y"]
X_test,  y_test  = test_df[FEATURES],  test_df["y"]
print(train_df.shape, test_df.shape, "event rate:", round(y_train.mean(), 3))


*Survival / time-to-event project? Use the alternative substrate below instead, and evaluate with concordance and time-dependent AUC rather than AUROC. (Imaging, text, or genomics: treat your encoder's output as the feature matrix and reuse the binary substrate.)*

In [ ]:
# --- Alternative substrate: time-to-event (survival) ---
# If your project is survival / time-to-event rather than binary classification,
# build on this instead of the generator above (pip install scikit-survival).
import numpy as np, pandas as pd

def make_synthetic_survival(n=4000, subgroup_gap=0.5, censor_rate=0.3, seed=0):
    """Right-censored survival data with a covariate-driven hazard and a
    subgroup with weaker signal — adapt the hazard to your project."""
    rng = np.random.default_rng(seed)
    subgroup = rng.integers(0, 2, n)
    x1 = rng.normal(0, 1, n)
    x2 = rng.normal(0, 1, n)
    risk = 0.8 * x1 + 0.4 * x2 + 0.5 * subgroup * (1 - subgroup_gap)
    event_time = rng.exponential(1 / np.clip(np.exp(risk - 1.0), 1e-3, None))
    censor_time = rng.exponential(scale=event_time.mean() / max(censor_rate, 1e-3), size=n)
    observed = np.minimum(event_time, censor_time)
    event = event_time <= censor_time
    return pd.DataFrame({"x1": x1, "x2": x2, "subgroup": subgroup,
                         "time": observed, "event": event})

surv_df = make_synthetic_survival()
print(surv_df.shape, "event rate:", round(surv_df.event.mean(), 3))
# scikit-survival models take a structured array: Surv.from_arrays(event, time).
# Evaluate with the concordance index and time-dependent AUC instead of AUROC.


## Section 1 — Discrimination

ROC/AUROC and precision–recall/AUPRC. Under realistic class imbalance, a strong AUROC can coexist with a model that is useless at the operating point. **Which of these tracks your decision?**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
p_test = clf.predict_proba(X_test)[:, 1]
print("AUROC:", round(roc_auc_score(y_test, p_test), 3))
print("AUPRC:", round(average_precision_score(y_test, p_test), 3))
# TODO: plot ROC and PR curves; relate AUPRC to the base rate.

## Section 2 — Calibration

A well-discriminating model can be badly miscalibrated. Calibration is what matters the moment a prediction informs a decision. Plot a reliability curve and compute a calibration error.

In [ ]:
from sklearn.calibration import calibration_curve
frac_pos, mean_pred = calibration_curve(y_test, p_test, n_bins=10, strategy="quantile")
# TODO: plot mean_pred vs frac_pos against the diagonal; report ECE.
print(list(zip(mean_pred.round(2), frac_pos.round(2))))

## Section 3 — Clinical utility

Decision-curve analysis / net benefit: choose an operating threshold from the **decision**, not the metric. Net benefit at threshold *t* = `TP/n - FP/n * (t/(1-t))`.

In [ ]:
import numpy as np

def net_benefit(y_true, p, t):
    pred = (p >= t).astype(int)
    tp = np.sum((pred == 1) & (y_true == 1))
    fp = np.sum((pred == 1) & (y_true == 0))
    n = len(y_true)
    return tp / n - fp / n * (t / (1 - t))

for t in [0.1, 0.2, 0.3, 0.5]:
    print(t, round(net_benefit(y_test.values, p_test, t), 4))
# TODO: plot net benefit vs treat-all / treat-none across thresholds.

## Section 4 — Disaggregated (subgroup / fairness) evaluation

Aggregate metrics hide subgroup failures. Report your metrics per-subgroup and name the gap that would concern you.

In [ ]:
for g, name in [(0, "A"), (1, "B")]:
    mask = test_df["subgroup"].values == g
    print(name, "AUROC:", round(roc_auc_score(y_test[mask], p_test[mask]), 3))
# TODO: add calibration and net benefit per subgroup; is the gap acceptable?

## Section 5 — Uncertainty & power

A small test set yields an estimate too noisy to act on. Bootstrap a confidence interval for your primary metric and reason about the test size you actually need.

In [ ]:
rng = np.random.default_rng(0)
boot = []
yt, pt = y_test.values, p_test
for _ in range(1000):
    idx = rng.integers(0, len(yt), len(yt))
    if yt[idx].sum() in (0, len(idx)):
        continue
    boot.append(roc_auc_score(yt[idx], pt[idx]))
print("AUROC 95% CI:", np.round(np.percentile(boot, [2.5, 97.5]), 3))

## Section 6 — Leakage & shortcut probing

A planted shortcut inflates in-distribution performance and collapses under shift. Compare a model **with** the shortcut feature against one **without** it, on the shifted test set.

In [ ]:
def auroc_with(features):
    m = LogisticRegression(max_iter=1000).fit(X_train[features], y_train)
    return roc_auc_score(y_test, m.predict_proba(X_test[features])[:, 1])

print("with shortcut:   ", round(auroc_with(["x1", "x2", "shortcut"]), 3))
print("without shortcut:", round(auroc_with(["x1", "x2"]), 3))
# A big gap means the model leaned on a feature that will not generalize.

## Section 7 — Findings → Workbook Part 3

*Convert what you found into your evaluation plan (Project Workbook Part 3):*

- **Estimand** — what, precisely, are you measuring?
- **Primary metric** and why it tracks the real decision (not convenience).
- **Secondary metrics** — calibration, net benefit, others.
- **Subgroup / fairness plan** — which subgroups; what gap concerns you.
- **Power** — how much test data you need for a usable estimate.
- **Top 3 threats to validity** (leakage, shortcut, shift) and your guard against each.